### Different test for consciousness

In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Load data
df_left = pd.read_csv("../Modified_Pupilometri/T50_left.csv")
df_right = pd.read_csv("../Modified_Pupilometri/T50_right.csv")
df_P = pd.concat([df_left, df_right], ignore_index=True)
df_pupil = pd.read_csv("../R_attempt/df_T50.csv")
df_pupil['record_id'] = df_pupil['pt_id'].str.replace('S_', '', regex=False)
df_pupil['record_id'] = df_pupil['record_id'].astype(int)
df_pupil['redcap_repeat_instance'] = df_pupil['day']
df_pupil['eye'] = df_pupil['lateral']
merged_df = df_P.merge(
    df_pupil,
    on=['record_id', 'redcap_repeat_instance', 'eye'],
    how='left'
)

### VIF check to ensure tests are valid

In [8]:
vif_data = df_pupil[["T50_constr", "T50_dilat", "day"]].dropna().copy()

# Standardize (This is also describe in the first code cell)
vif_data["z_PLR"] = (vif_data["T50_constr"] - 3.57) / 0.089
vif_data["z_LOR"] = (vif_data["T50_dilat"] - 8.02) / 0.1

vif_matrix = vif_data[["z_PLR", "z_LOR", "day"]].reset_index(drop=True)

vif_results = pd.DataFrame({
    "Predictor": vif_matrix.columns,
    "VIF": [
        variance_inflation_factor(vif_matrix.values, i)
        for i in range(vif_matrix.shape[1])
    ]
})

vif_results["Concern"] = vif_results["VIF"].apply(
    lambda v: "None" if v < 5 else ("Moderate" if v < 10 else "High — consider removing")
)

# Dont incluede consciouseness measures because these are recoreded indepentendly of the pupil recording
print(vif_results.to_string(index=False))


Predictor      VIF Concern
    z_PLR 1.155166    None
    z_LOR 2.625053    None
      day 2.396442    None


# Using repeat_instance as time index

## FOUR

### 7-day measurement (T7)

In [9]:
df_future = df_pupil.copy()
df_future["day"] = df_future["day"] - 7 # Time index is here (different in each code cell based on what instance to use)

df_t7 = pd.merge(
    df_pupil,
    df_future[["pt_id", "day", "FOUR",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t7")
)

# Create dataframe based on conscious measure and time index (Using FOUR, GCS and SECONDS in the different cells)
df_t7 = df_t7[['pt_id','day','lateral','T50_constr','T50_dilat','FOUR','FOUR_t7']]
df_t7 = df_t7.dropna().reset_index(drop=True)

# Standadize with the healthy mean and std (This is done in all code blocks)
z_PLR = (
    df_t7['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t7['T50_dilat'] - 8.02
) / 0.1

df_t7[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_t7 ~ FOUR+ z_PLR + z_LOR + day",
    data=df_t7,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: FOUR_t7  
No. Observations: 204     Method:             REML     
No. Groups:       34      Scale:              4.7070   
Min. group size:  2       Log-Likelihood:     -497.4181
Max. group size:  10      Converged:          Yes      
Mean group size:  6.0                                  
-------------------------------------------------------
              Coef.  Std.Err.   z   P>|z| [0.025 0.975]
-------------------------------------------------------
Intercept      6.373    1.220 5.225 0.000  3.982  8.763
FOUR           0.137    0.111 1.234 0.217 -0.081  0.355
z_PLR          0.008    0.024 0.336 0.737 -0.038  0.054
z_LOR          0.045    0.033 1.356 0.175 -0.020  0.110
day            0.954    0.142 6.718 0.000  0.675  1.232
pt_id Var     12.008    1.688                          



### 3-day measurement (T3)

In [10]:
df_future = df_pupil.copy()
df_future["day"] = df_future["day"] - 3

df_t3 = pd.merge(
    df_pupil,
    df_future[["pt_id", "day", "FOUR",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t3")
)

df_t3 = df_t3[['pt_id','day','lateral','T50_constr','T50_dilat','FOUR','FOUR_t3']]
df_t3 = df_t3.dropna().reset_index(drop=True)


z_PLR = (
    df_t3['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t3['T50_dilat'] - 8.02
) / 0.1

df_t3[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_t3 ~ FOUR+ z_PLR + z_LOR + day",
    data=df_t3,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: FOUR_t3   
No. Observations: 763     Method:             REML      
No. Groups:       104     Scale:              4.3456    
Min. group size:  2       Log-Likelihood:     -1805.4426
Max. group size:  18      Converged:          Yes       
Mean group size:  7.3                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      6.741    0.576 11.700 0.000  5.612  7.870
FOUR           0.024    0.048  0.496 0.620 -0.070  0.117
z_PLR          0.002    0.012  0.127 0.899 -0.022  0.025
z_LOR          0.008    0.013  0.656 0.512 -0.017  0.034
day            0.877    0.051 17.354 0.000  0.778  0.976
pt_id Var     13.884    1.129                           



### 1-day measurement (T1)

In [11]:
df_future = df_pupil.copy()
df_future["day"] = df_future["day"] - 1

df_t1 = pd.merge(
    df_pupil,
    df_future[["pt_id", "day", "FOUR",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t1")
)

df_t1 = df_t1[['pt_id','day','lateral','T50_constr','T50_dilat','FOUR','FOUR_t1']]
df_t1 = df_t1.dropna().reset_index(drop=True)


z_PLR = (
    df_t1['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t1['T50_dilat'] - 8.02
) / 0.1

df_t1[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_t1 ~ FOUR+ z_PLR + z_LOR + day",
    data=df_t1,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: FOUR_t1   
No. Observations: 1417    Method:             REML      
No. Groups:       188     Scale:              5.1032    
Min. group size:  2       Log-Likelihood:     -3435.4546
Max. group size:  22      Converged:          Yes       
Mean group size:  7.5                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      5.633    0.412 13.656 0.000  4.824  6.441
FOUR           0.254    0.031  8.140 0.000  0.193  0.316
z_PLR          0.003    0.010  0.285 0.776 -0.017  0.023
z_LOR         -0.006    0.010 -0.601 0.548 -0.025  0.013
day            0.673    0.039 17.307 0.000  0.597  0.749
pt_id Var     13.136    0.761                           



### Current day measurement (T0)

In [12]:
df_t0 = merged_df[['pt_id','day','lateral','T50_constr','T50_dilat','FOUR_x']]
df_t0 = df_t0.dropna().reset_index(drop=True)


z_PLR = (
    df_t0['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t0['T50_dilat'] - 8.02
) / 0.1

df_t0[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_x ~  z_PLR + z_LOR + day",
    data=df_t0,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: FOUR_x    
No. Observations: 1905    Method:             REML      
No. Groups:       250     Scale:              6.5021    
Min. group size:  1       Log-Likelihood:     -4756.6964
Max. group size:  24      Converged:          Yes       
Mean group size:  7.6                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      5.077    0.314 16.156 0.000  4.461  5.692
z_PLR         -0.075    0.010 -7.660 0.000 -0.094 -0.056
z_LOR         -0.004    0.009 -0.387 0.699 -0.021  0.014
day            0.800    0.030 26.952 0.000  0.742  0.858
pt_id Var      7.823    0.348                           



## GCS

### Current day measurement (T0)

In [13]:
df_t0 = merged_df[['pt_id','day','lateral','T50_constr','T50_dilat','GCS_x']]
df_t0 = df_t0.dropna().reset_index(drop=True)


z_PLR = (
    df_t0['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t0['T50_dilat'] - 8.02
) / 0.1

df_t0[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_x ~  z_PLR + z_LOR + day",
    data=df_t0,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: GCS_x     
No. Observations: 1905    Method:             REML      
No. Groups:       250     Scale:              4.9487    
Min. group size:  1       Log-Likelihood:     -4469.2164
Max. group size:  24      Converged:          Yes       
Mean group size:  7.6                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      4.363    0.263 16.594 0.000  3.848  4.878
z_PLR         -0.063    0.008 -7.507 0.000 -0.080 -0.047
z_LOR         -0.001    0.008 -0.127 0.899 -0.017  0.015
day            0.559    0.026 21.609 0.000  0.508  0.610
pt_id Var      4.541    0.237                           



### 1-day measurement (T1)

In [14]:
df_future = df_pupil.copy()
df_future["day"] = df_future["day"] - 1

df_t1 = pd.merge(
    df_pupil,
    df_future[["pt_id", "day", "GCS",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t1")
)

df_t1 = df_t1[['pt_id','day','lateral','T50_constr','T50_dilat','GCS','GCS_t1']]
df_t1 = df_t1.dropna().reset_index(drop=True)


z_PLR = (
    df_t1['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t1['T50_dilat'] - 8.02
) / 0.1

df_t1[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_t1 ~ GCS+ z_PLR + z_LOR + day",
    data=df_t1,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: GCS_t1    
No. Observations: 1417    Method:             REML      
No. Groups:       188     Scale:              3.9560    
Min. group size:  2       Log-Likelihood:     -3251.3039
Max. group size:  22      Converged:          Yes       
Mean group size:  7.5                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      4.613    0.357 12.910 0.000  3.913  5.314
GCS            0.248    0.032  7.736 0.000  0.185  0.311
z_PLR         -0.001    0.009 -0.166 0.868 -0.019  0.016
z_LOR         -0.003    0.009 -0.377 0.706 -0.020  0.014
day            0.499    0.032 15.585 0.000  0.436  0.562
pt_id Var      9.694    0.636                           



### 3-day measurement (T3)

In [15]:
df_future = df_pupil.copy()
df_future["day"] = df_future["day"] - 3

df_t3 = pd.merge(
    df_pupil,
    df_future[["pt_id", "day", "GCS",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t3")
)

df_t3 = df_t3[['pt_id','day','lateral','T50_constr','T50_dilat','GCS','GCS_t3']]
df_t3 = df_t3.dropna().reset_index(drop=True)


z_PLR = (
    df_t3['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t3['T50_dilat'] - 8.02
) / 0.1

df_t3[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_t3 ~ GCS+ z_PLR + z_LOR + day",
    data=df_t3,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: GCS_t3    
No. Observations: 763     Method:             REML      
No. Groups:       104     Scale:              3.4240    
Min. group size:  2       Log-Likelihood:     -1715.4767
Max. group size:  18      Converged:          Yes       
Mean group size:  7.3                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      5.652    0.506 11.178 0.000  4.661  6.643
GCS           -0.018    0.050 -0.367 0.713 -0.117  0.080
z_PLR          0.002    0.011  0.196 0.845 -0.019  0.023
z_LOR          0.001    0.011  0.097 0.922 -0.021  0.024
day            0.664    0.043 15.405 0.000  0.580  0.749
pt_id Var     11.066    1.010                           



### 7-day measurement (T7)

In [16]:
df_future = df_pupil.copy()
df_future["day"] = df_future["day"] - 7

df_t7 = pd.merge(
    df_pupil,
    df_future[["pt_id", "day", "GCS",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t7")
)

df_t7 = df_t7[['pt_id','day','lateral','T50_constr','T50_dilat','GCS','GCS_t7']]
df_t7 = df_t7.dropna().reset_index(drop=True)


z_PLR = (
    df_t7['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t7['T50_dilat'] - 8.02
) / 0.1

df_t7[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_t7 ~ GCS+ z_PLR + z_LOR + day",
    data=df_t7,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: GCS_t7   
No. Observations: 204     Method:             REML     
No. Groups:       34      Scale:              3.2930   
Min. group size:  2       Log-Likelihood:     -458.3690
Max. group size:  10      Converged:          Yes      
Mean group size:  6.0                                  
-------------------------------------------------------
             Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------
Intercept     4.797    0.971  4.940 0.000  2.894  6.700
GCS           0.043    0.115  0.370 0.711 -0.183  0.268
z_PLR        -0.020    0.020 -0.999 0.318 -0.058  0.019
z_LOR         0.040    0.028  1.433 0.152 -0.015  0.094
day           0.770    0.118  6.512 0.000  0.538  1.002
pt_id Var     6.740    1.134                           



## SECONDS


### 7-day measurement (T7)

In [17]:
df_future = merged_df.copy()
df_future = df_future[['pt_id','day','lateral','T50_constr','T50_dilat','SECONDS']].dropna().reset_index(drop=True)
df_future["day"] = df_future["day"] - 7

df_t7 = pd.merge(
    merged_df,
    df_future[["pt_id", "day", "SECONDS",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t7")
)

df_t7 = df_t7[['pt_id','day','lateral','T50_constr','T50_dilat','SECONDS','SECONDS_t7']]
df_t7 = df_t7.dropna().reset_index(drop=True)

# Map SECONDS to a numeric value
mapping_numeric = {
    "C": 0,
    "U": 1,
    "M-": 2,
    "M+": 3,
    "E": 4
}
df_t7["SECONDS"] = df_t7["SECONDS"].map(mapping_numeric)
df_t7["SECONDS_t7"] = df_t7["SECONDS_t7"].map(mapping_numeric)

z_PLR = (
    df_t7['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t7['T50_dilat'] - 8.02
) / 0.1

df_t7[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS_t7 ~ SECONDS+ z_PLR + z_LOR + day",
    data=df_t7,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: SECONDS_t7
No. Observations: 202     Method:             REML      
No. Groups:       34      Scale:              0.5327    
Min. group size:  2       Log-Likelihood:     -277.9561 
Max. group size:  10      Converged:          Yes       
Mean group size:  5.9                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      0.701    0.370  1.896 0.058 -0.024  1.425
SECONDS        0.429    0.149  2.880 0.004  0.137  0.721
z_PLR         -0.016    0.008 -2.031 0.042 -0.032 -0.001
z_LOR          0.014    0.011  1.234 0.217 -0.008  0.036
day            0.271    0.049  5.539 0.000  0.175  0.367
pt_id Var      1.458    0.598                           



### 3-day measurement (T3)

In [18]:
df_future = merged_df.copy()
df_future = df_future[['pt_id','day','lateral','T50_constr','T50_dilat','SECONDS']].dropna().reset_index(drop=True)
df_future["day"] = df_future["day"] - 3

df_t3 = pd.merge(
    merged_df,
    df_future[["pt_id", "day", "SECONDS",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t3")
)

df_t3 = df_t3[['pt_id','day','lateral','T50_constr','T50_dilat','SECONDS','SECONDS_t3']]
df_t3= df_t3.dropna().reset_index(drop=True)

mapping_numeric = {
    "C": 0,
    "U": 1,
    "M-": 2,
    "M+": 3,
    "E": 4
}
df_t3["SECONDS"] = df_t3["SECONDS"].map(mapping_numeric)
df_t3["SECONDS_t3"] = df_t3["SECONDS_t3"].map(mapping_numeric)

z_PLR = (
    df_t3['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t3['T50_dilat'] - 8.02
) / 0.1

df_t3[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS_t3 ~ SECONDS+ z_PLR + z_LOR + day",
    data=df_t3,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: SECONDS_t3
No. Observations: 754     Method:             REML      
No. Groups:       104     Scale:              0.5258    
Min. group size:  2       Log-Likelihood:     -991.4663 
Max. group size:  18      Converged:          Yes       
Mean group size:  7.2                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      0.676    0.180  3.746 0.000  0.322  1.029
SECONDS        0.139    0.053  2.646 0.008  0.036  0.242
z_PLR         -0.009    0.004 -2.009 0.044 -0.017 -0.000
z_LOR          0.006    0.005  1.395 0.163 -0.003  0.015
day            0.230    0.017 13.729 0.000  0.197  0.263
pt_id Var      1.611    0.377                           



### 1-day measurement (T1)

In [19]:
df_future = merged_df.copy()
df_future = df_future[['pt_id','day','lateral','T50_constr','T50_dilat','SECONDS']].dropna().reset_index(drop=True)
df_future["day"] = df_future["day"] - 1

df_t1 = pd.merge(
    merged_df,
    df_future[["pt_id", "day", "SECONDS",'lateral']],
    on=["pt_id", "day",'lateral'],  
    how="inner",
    suffixes=("", "_t1")
)

df_t1 = df_t1[['pt_id','day','lateral','T50_constr','T50_dilat','SECONDS','SECONDS_t1']]
df_t1= df_t1.dropna().reset_index(drop=True)

mapping_numeric = {
    "C": 0,
    "U": 1,
    "M-": 2,
    "M+": 3,
    "E": 4
}
df_t1["SECONDS"] = df_t1["SECONDS"].map(mapping_numeric)
df_t1["SECONDS_t1"] = df_t1["SECONDS_t1"].map(mapping_numeric)

z_PLR = (
    df_t1['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t1['T50_dilat'] - 8.02
) / 0.1

df_t1[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS_t1 ~ SECONDS+ z_PLR + z_LOR + day",
    data=df_t1,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: SECONDS_t1
No. Observations: 1400    Method:             REML      
No. Groups:       188     Scale:              0.6001    
Min. group size:  1       Log-Likelihood:     -1886.5298
Max. group size:  22      Converged:          Yes       
Mean group size:  7.4                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      0.761    0.126  6.031 0.000  0.514  1.008
SECONDS        0.321    0.033  9.605 0.000  0.255  0.386
z_PLR         -0.005    0.003 -1.373 0.170 -0.012  0.002
z_LOR         -0.002    0.003 -0.502 0.616 -0.008  0.005
day            0.171    0.012 13.820 0.000  0.147  0.195
pt_id Var      1.289    0.223                           



### Current day measurement (T0)

In [20]:

df_t0 = merged_df[['pt_id','day','lateral','T50_constr','T50_dilat','SECONDS']].dropna().reset_index(drop=True)

mapping_numeric = {
    "C": 0,
    "U": 1,
    "M-": 2,
    "M+": 3,
    "E": 4
}
df_t0["SECONDS"] = df_t0["SECONDS"].map(mapping_numeric)

z_PLR = (
    df_t0['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t0['T50_dilat'] - 8.02
) / 0.1

df_t0[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS ~ z_PLR + z_LOR + day",
    data=df_t0,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: SECONDS   
No. Observations: 1905    Method:             REML      
No. Groups:       250     Scale:              0.7330    
Min. group size:  1       Log-Likelihood:     -2648.8048
Max. group size:  24      Converged:          Yes       
Mean group size:  7.6                                   
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      0.325    0.101  3.230 0.001  0.128  0.522
z_PLR         -0.019    0.003 -5.805 0.000 -0.025 -0.013
z_LOR          0.002    0.003  0.494 0.621 -0.004  0.008
day            0.206    0.010 20.645 0.000  0.186  0.225
pt_id Var      0.639    0.087                           



### Bonferroni test of p-values 

In [3]:
from statsmodels.stats.multitest import multipletests

# Raw p-values from repeat instance models 

results = pd.DataFrame([
    {"outcome": "FOUR",    "horizon": "t0", "p_PLR": 0.000, "p_LOR": 0.699},
    {"outcome": "FOUR",    "horizon": "t1", "p_PLR": 0.776, "p_LOR": 0.548},
    {"outcome": "FOUR",    "horizon": "t3", "p_PLR": 0.899, "p_LOR": 0.512},
    {"outcome": "FOUR",    "horizon": "t7", "p_PLR": 0.737, "p_LOR": 0.175},
    {"outcome": "GCS",     "horizon": "t0", "p_PLR": 0.000, "p_LOR": 0.899},
    {"outcome": "GCS",     "horizon": "t1", "p_PLR": 0.868, "p_LOR": 0.706},
    {"outcome": "GCS",     "horizon": "t3", "p_PLR": 0.845, "p_LOR": 0.922},
    {"outcome": "GCS",     "horizon": "t7", "p_PLR": 0.318, "p_LOR": 0.152},
    {"outcome": "SECONDS", "horizon": "t0", "p_PLR": 0.000, "p_LOR": 0.621},
    {"outcome": "SECONDS", "horizon": "t1", "p_PLR": 0.170, "p_LOR": 0.616},
    {"outcome": "SECONDS", "horizon": "t3", "p_PLR": 0.044, "p_LOR": 0.163},
    {"outcome": "SECONDS", "horizon": "t7", "p_PLR": 0.042, "p_LOR": 0.217},
])

# Apply Bonferroni correction 
for predictor in ["PLR", "LOR"]:
    col = f"p_{predictor}"
    _, p_corrected, _, _ = multipletests(results[col], alpha=0.05, method="bonferroni")
    results[f"p_{predictor}_corrected"] = p_corrected
    results[f"{predictor}_significant"] = p_corrected < 0.05

n_tests = len(results)
alpha_corrected = 0.05 / n_tests

# Print in a nice way
print("=" * 65)
print("Bonferroni correction — repeat instance")
print(f"Outcomes: FOUR, GCS, SECONDS  |  Horizons: t0, t1, t3, t7")
print(f"Total tests per predictor: {n_tests}  |  Corrected α: {alpha_corrected:.4f}")
print("=" * 65)

for predictor in ["LOR", "PLR"]:
    print(f"\nz_{predictor}")
    print("-" * 55)
    print(f"  {'Outcome':<10} {'Horizon':<10} {'Raw p':<12} {'Corrected p':<14} {'Sig?'}")
    print(f"  {'-'*8:<10} {'-'*7:<10} {'-'*9:<12} {'-'*11:<14} {'-'*4}")
    for _, row in results.iterrows():
        raw      = row[f"p_{predictor}"]
        corrected = row[f"p_{predictor}_corrected"]
        sig      = row[f"{predictor}_significant"]
        raw_str  = "< 0.001" if raw < 0.001 else f"{raw:.3f}"
        cor_str  = "< 0.001" if corrected < 0.001 else f"{corrected:.3f}"
        sig_str  = "YES *" if sig else "no"
        print(f"  {row['outcome']:<10} {row['horizon']:<10} {raw_str:<12} {cor_str:<14} {sig_str}")

print("\n" + "=" * 65)
print("Summary")
print("-" * 65)
for predictor in ["LOR", "PLR"]:
    n_sig = results[f"{predictor}_significant"].sum()
    print(f"  z_{predictor}: {n_sig}/{n_tests} significant after correction")
print("=" * 65)

Bonferroni correction — repeat instance
Outcomes: FOUR, GCS, SECONDS  |  Horizons: t0, t1, t3, t7
Total tests per predictor: 12  |  Corrected α: 0.0042

z_LOR
-------------------------------------------------------
  Outcome    Horizon    Raw p        Corrected p    Sig?
  --------   -------    ---------    -----------    ----
  FOUR       t0         0.699        1.000          no
  FOUR       t1         0.548        1.000          no
  FOUR       t3         0.512        1.000          no
  FOUR       t7         0.175        1.000          no
  GCS        t0         0.899        1.000          no
  GCS        t1         0.706        1.000          no
  GCS        t3         0.922        1.000          no
  GCS        t7         0.152        1.000          no
  SECONDS    t0         0.621        1.000          no
  SECONDS    t1         0.616        1.000          no
  SECONDS    t3         0.163        1.000          no
  SECONDS    t7         0.217        1.000          no

z_PLR
----

# Using real days (dates) as time index

In [22]:
merged_df["date_examination_merged"] = pd.to_datetime(merged_df["date_examination_merged"])
merged_df["days_since_first"] = (
    merged_df.groupby("record_id")["date_examination_merged"]
    .transform(lambda x: (x - x.min()).dt.days)
)

## FOUR

### 7-day measurement (T7)

In [23]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','FOUR_x']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 7  # shift backward

df_t7 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "FOUR_x",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t7")
)

df_t7 = df_t7[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','FOUR_x','FOUR_x_t7']]
df_t7 = df_t7.dropna().reset_index(drop=True)


z_PLR = (
    df_t7['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t7['T50_dilat'] - 8.02
) / 0.1

df_t7[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_x_t7 ~ FOUR_x+ z_PLR + z_LOR + days_since_first",
    data=df_t7,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  FOUR_x_t7
No. Observations:   402      Method:              REML     
No. Groups:         80       Scale:               3.1274   
Min. group size:    2        Log-Likelihood:      -934.4740
Max. group size:    14       Converged:           Yes      
Mean group size:    5.0                                    
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         8.095    0.737 10.979 0.000  6.650  9.541
FOUR_x           -0.096    0.067 -1.439 0.150 -0.226  0.035
z_PLR            -0.020    0.014 -1.362 0.173 -0.048  0.009
z_LOR            -0.007    0.017 -0.413 0.679 -0.039  0.026
days_since_first  0.431    0.032 13.416 0.000  0.368  0.494
pt_id Var        17.367    1.982                           



### 1-day measurement (T1)

In [24]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','FOUR_x']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 1  # shift backward

df_t1 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "FOUR_x",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t1")
)

df_t1 = df_t1[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','FOUR_x','FOUR_x_t1']]
df_t1 = df_t1.dropna().reset_index(drop=True)


z_PLR = (
    df_t1['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t1['T50_dilat'] - 8.02
) / 0.1

df_t1[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_x_t1 ~ FOUR_x+ z_PLR + z_LOR + days_since_first",
    data=df_t1,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  FOUR_x_t1 
No. Observations:  829      Method:              REML      
No. Groups:        157      Scale:               3.9722    
Min. group size:   1        Log-Likelihood:      -1941.8701
Max. group size:   16       Converged:           Yes       
Mean group size:   5.3                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         5.060    0.446 11.333 0.000  4.185  5.935
FOUR_x            0.378    0.039  9.718 0.000  0.302  0.454
z_PLR            -0.009    0.012 -0.738 0.460 -0.033  0.015
z_LOR            -0.023    0.012 -1.943 0.052 -0.046  0.000
days_since_first  0.247    0.023 10.783 0.000  0.202  0.292
pt_id Var         8.649    0.702                           



### 3-day measurement (T3)

In [25]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','FOUR_x']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 3  # shift backward

df_t3 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "FOUR_x",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t3")
)

df_t3 = df_t3[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','FOUR_x','FOUR_x_t3']]
df_t3 = df_t3.dropna().reset_index(drop=True)


z_PLR = (
    df_t3['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t3['T50_dilat'] - 8.02
) / 0.1

df_t3[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_x_t3 ~ FOUR_x+ z_PLR + z_LOR + days_since_first",
    data=df_t3,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  FOUR_x_t3 
No. Observations:  431      Method:              REML      
No. Groups:        102      Scale:               3.4942    
Min. group size:   1        Log-Likelihood:      -1017.1982
Max. group size:   13       Converged:           Yes       
Mean group size:   4.2                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         6.541    0.618 10.588 0.000  5.330  7.752
FOUR_x            0.322    0.055  5.902 0.000  0.215  0.429
z_PLR             0.002    0.016  0.113 0.910 -0.029  0.032
z_LOR            -0.029    0.016 -1.854 0.064 -0.059  0.002
days_since_first  0.224    0.030  7.396 0.000  0.164  0.283
pt_id Var        11.109    1.171                           



### Current day measurement (T0)

In [26]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','FOUR_x']].dropna()

z_PLR = (
    copy['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    copy['T50_dilat'] - 8.02
) / 0.1

copy[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "FOUR_x ~ z_PLR + z_LOR + days_since_first",
    data=copy,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  FOUR_x    
No. Observations:  1903     Method:              REML      
No. Groups:        249      Scale:               6.1994    
Min. group size:   1        Log-Likelihood:      -4715.4344
Max. group size:   24       Converged:           Yes       
Mean group size:   7.6                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         5.851    0.302 19.381 0.000  5.259  6.443
z_PLR            -0.070    0.010 -7.324 0.000 -0.089 -0.051
z_LOR            -0.002    0.009 -0.182 0.855 -0.019  0.016
days_since_first  0.357    0.012 28.909 0.000  0.332  0.381
pt_id Var         8.080    0.366                           



## SECONDS


### 7-day measurement (T7)

In [27]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','SECONDS']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 7  # shift backward

df_t7 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "SECONDS",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t7")
)

df_t7 = df_t7[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','SECONDS','SECONDS_t7']]
df_t7 = df_t7.dropna().reset_index(drop=True)


z_PLR = (
    df_t7['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t7['T50_dilat'] - 8.02
) / 0.1

df_t7[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
mapping_numeric = {
    "C": 0,
    "U": 1,
    "M-": 2,
    "M+": 3,
    "E": 4
}
df_t7["SECONDS"] = df_t7["SECONDS"].map(mapping_numeric)
df_t7["SECONDS_t7"] = df_t7["SECONDS_t7"].map(mapping_numeric)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS_t7 ~ SECONDS + z_PLR + z_LOR + days_since_first",
    data=df_t7,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  SECONDS_t7
No. Observations:  402      Method:              REML      
No. Groups:        80       Scale:               0.4074    
Min. group size:   2        Log-Likelihood:      -518.6346 
Max. group size:   14       Converged:           Yes       
Mean group size:   5.0                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         0.975    0.224  4.362 0.000  0.537  1.413
SECONDS           0.038    0.075  0.506 0.613 -0.109  0.185
z_PLR            -0.004    0.005 -0.768 0.443 -0.014  0.006
z_LOR            -0.003    0.006 -0.504 0.614 -0.015  0.009
days_since_first  0.110    0.011 10.345 0.000  0.089  0.131
pt_id Var         1.724    0.554                           



### 3-day measurement (T3)

In [28]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','SECONDS']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 3  # shift backward

df_t3 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "SECONDS",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t3")
)

df_t3 = df_t3[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','SECONDS','SECONDS_t3']]
df_t3 = df_t3.dropna().reset_index(drop=True)


z_PLR = (
    df_t3['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t3['T50_dilat'] - 8.02
) / 0.1

df_t3[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)

df_t3["SECONDS"] = df_t3["SECONDS"].map(mapping_numeric)
df_t3["SECONDS_t3"] = df_t3["SECONDS_t3"].map(mapping_numeric)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS_t3 ~ SECONDS + z_PLR + z_LOR + days_since_first",
    data=df_t3,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  SECONDS_t3
No. Observations:  431      Method:              REML      
No. Groups:        102      Scale:               0.4462    
Min. group size:   1        Log-Likelihood:      -570.3140 
Max. group size:   13       Converged:           Yes       
Mean group size:   4.2                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         0.889    0.190  4.670 0.000  0.516  1.262
SECONDS           0.475    0.058  8.199 0.000  0.362  0.589
z_PLR            -0.002    0.006 -0.358 0.720 -0.013  0.009
z_LOR            -0.002    0.006 -0.289 0.773 -0.013  0.009
days_since_first  0.052    0.010  5.413 0.000  0.033  0.071
pt_id Var         1.204    0.356                           



### 1-day measurement (T1)

In [29]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','SECONDS']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 1  # shift backward

df_t1 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "SECONDS",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t1")
)

df_t1 = df_t1[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','SECONDS','SECONDS_t1']]
df_t1 = df_t1.dropna().reset_index(drop=True)


z_PLR = (
    df_t1['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t1['T50_dilat'] - 8.02
) / 0.1

df_t1[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)

df_t1["SECONDS"] = df_t1["SECONDS"].map(mapping_numeric)
df_t1["SECONDS_t1"] = df_t1["SECONDS_t1"].map(mapping_numeric)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS_t1 ~ SECONDS + z_PLR + z_LOR + days_since_first",
    data=df_t1,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  SECONDS_t1
No. Observations:  829      Method:              REML      
No. Groups:        157      Scale:               0.5145    
Min. group size:   1        Log-Likelihood:      -1077.3567
Max. group size:   16       Converged:           Yes       
Mean group size:   5.3                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         0.858    0.135  6.337 0.000  0.593  1.124
SECONDS           0.404    0.041  9.915 0.000  0.324  0.484
z_PLR            -0.008    0.004 -1.917 0.055 -0.017  0.000
z_LOR            -0.011    0.004 -2.642 0.008 -0.019 -0.003
days_since_first  0.058    0.007  8.025 0.000  0.044  0.073
pt_id Var         0.817    0.191                           



### Current day measurement (T0)

In [30]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','SECONDS']].dropna()


z_PLR = (
    copy['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    copy['T50_dilat'] - 8.02
) / 0.1

copy[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)

copy["SECONDS"] = copy["SECONDS"].map(mapping_numeric)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "SECONDS ~  z_PLR + z_LOR + days_since_first",
    data=copy,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  SECONDS   
No. Observations:  1903     Method:              REML      
No. Groups:        249      Scale:               0.7396    
Min. group size:   1        Log-Likelihood:      -2655.2576
Max. group size:   24       Converged:           Yes       
Mean group size:   7.6                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         0.561    0.098  5.750 0.000  0.370  0.752
z_PLR            -0.019    0.003 -5.867 0.000 -0.026 -0.013
z_LOR             0.002    0.003  0.498 0.618 -0.004  0.008
days_since_first  0.085    0.004 20.119 0.000  0.077  0.094
pt_id Var         0.645    0.088                           



## GCS

### 7-day measurement (T7)

In [31]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','GCS_x']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 7  # shift backward

df_t7 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "GCS_x",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t7")
)

df_t7 = df_t7[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','GCS_x','GCS_x_t7']]
df_t7 = df_t7.dropna().reset_index(drop=True)


z_PLR = (
    df_t7['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t7['T50_dilat'] - 8.02
) / 0.1

df_t7[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_x_t7 ~ GCS_x+ z_PLR + z_LOR + days_since_first",
    data=df_t7,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  GCS_x_t7 
No. Observations:   402      Method:              REML     
No. Groups:         80       Scale:               2.3523   
Min. group size:    2        Log-Likelihood:      -875.8008
Max. group size:    14       Converged:           Yes      
Mean group size:    5.0                                    
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         6.021    0.616  9.770 0.000  4.813  7.228
GCS_x             0.039    0.064  0.603 0.547 -0.087  0.164
z_PLR            -0.017    0.013 -1.331 0.183 -0.041  0.008
z_LOR            -0.013    0.014 -0.874 0.382 -0.041  0.016
days_since_first  0.300    0.026 11.434 0.000  0.249  0.352
pt_id Var        12.380    1.600                           



### 3-day measurement (T3)

In [32]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','GCS_x']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 3  # shift backward

df_t3 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "GCS_x",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t3")
)

df_t3 = df_t3[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','GCS_x','GCS_x_t3']]
df_t3 = df_t3.dropna().reset_index(drop=True)


z_PLR = (
    df_t3['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t3['T50_dilat'] - 8.02
) / 0.1

df_t3[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_x_t3 ~ GCS_x+ z_PLR + z_LOR + days_since_first",
    data=df_t3,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  GCS_x_t3 
No. Observations:   431      Method:              REML     
No. Groups:         102      Scale:               3.0772   
Min. group size:    1        Log-Likelihood:      -984.9379
Max. group size:    13       Converged:           Yes      
Mean group size:    4.2                                    
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         5.617    0.567  9.915 0.000  4.507  6.727
GCS_x             0.243    0.061  3.959 0.000  0.123  0.364
z_PLR            -0.006    0.015 -0.411 0.681 -0.034  0.023
z_LOR            -0.023    0.015 -1.609 0.108 -0.052  0.005
days_since_first  0.183    0.027  6.915 0.000  0.131  0.235
pt_id Var         8.756    1.004                           



### 1-day measurement (T1)

In [33]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','GCS_x']].dropna()
df_future = copy.copy()
df_future["days_since_first"] = df_future["days_since_first"] - 1  # shift backward

df_t1 = pd.merge(
    copy,
    df_future[["pt_id", "days_since_first", "GCS_x",'lateral']],
    on=["pt_id", "days_since_first",'lateral'],  
    how="inner",
    suffixes=("", "_t1")
)

df_t1 = df_t1[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','GCS_x','GCS_x_t1']]
df_t1 = df_t1.dropna().reset_index(drop=True)


z_PLR = (
    df_t1['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t1['T50_dilat'] - 8.02
) / 0.1

df_t1[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_x_t1 ~ GCS_x+ z_PLR + z_LOR + days_since_first",
    data=df_t1,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  GCS_x_t1  
No. Observations:  829      Method:              REML      
No. Groups:        157      Scale:               3.1745    
Min. group size:   1        Log-Likelihood:      -1836.8593
Max. group size:   16       Converged:           Yes       
Mean group size:   5.3                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         4.134    0.382 10.824 0.000  3.385  4.882
GCS_x             0.408    0.039 10.380 0.000  0.331  0.485
z_PLR            -0.001    0.011 -0.088 0.930 -0.022  0.020
z_LOR            -0.026    0.011 -2.509 0.012 -0.047 -0.006
days_since_first  0.170    0.019  9.143 0.000  0.134  0.207
pt_id Var         5.755    0.526                           



### Current day measurement (T0)

In [34]:
copy = merged_df[['pt_id','days_since_first','lateral','T50_constr','T50_dilat','GCS_x']].dropna()
df_t1 = copy.dropna().reset_index(drop=True)


z_PLR = (
    df_t1['T50_constr'] - 3.57
) / 0.089

z_LOR = (
    df_t1['T50_dilat'] - 8.02
) / 0.1

df_t1[['z_PLR', 'z_LOR']] = pd.concat([z_PLR, z_LOR], axis=1)
# TRy to do a mixed effect model.
md = smf.mixedlm(
    "GCS_x ~  z_PLR + z_LOR + days_since_first",
    data=df_t1,
    groups="pt_id"
)

mdf = md.fit()
print(mdf.summary())


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  GCS_x     
No. Observations:  1903     Method:              REML      
No. Groups:        249      Scale:               4.9173    
Min. group size:   1        Log-Likelihood:      -4460.9487
Max. group size:   24       Converged:           Yes       
Mean group size:   7.6                                     
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         4.964    0.254 19.529 0.000  4.465  5.462
z_PLR            -0.063    0.008 -7.433 0.000 -0.079 -0.046
z_LOR            -0.000    0.008 -0.053 0.958 -0.016  0.015
days_since_first  0.239    0.011 21.858 0.000  0.218  0.261
pt_id Var         4.597    0.241                           



### Bonferroni

In [4]:
#  Raw p-values from actual days models

 
results = pd.DataFrame([
    {"outcome": "FOUR",    "horizon": "t0", "p_PLR": 0.000, "p_LOR": 0.855},
    {"outcome": "FOUR",    "horizon": "t1", "p_PLR": 0.460, "p_LOR": 0.052},
    {"outcome": "FOUR",    "horizon": "t3", "p_PLR": 0.910, "p_LOR": 0.064},
    {"outcome": "FOUR",    "horizon": "t7", "p_PLR": 0.173, "p_LOR": 0.679},
    {"outcome": "GCS",     "horizon": "t0", "p_PLR": 0.000, "p_LOR": 0.958},
    {"outcome": "GCS",     "horizon": "t1", "p_PLR": 0.930, "p_LOR": 0.012},
    {"outcome": "GCS",     "horizon": "t3", "p_PLR": 0.681, "p_LOR": 0.108},
    {"outcome": "GCS",     "horizon": "t7", "p_PLR": 0.183, "p_LOR": 0.382},
    {"outcome": "SECONDS", "horizon": "t0", "p_PLR": 0.000, "p_LOR": 0.618},
    {"outcome": "SECONDS", "horizon": "t1", "p_PLR": 0.055, "p_LOR": 0.008},
    {"outcome": "SECONDS", "horizon": "t3", "p_PLR": 0.720, "p_LOR": 0.773},
    {"outcome": "SECONDS", "horizon": "t7", "p_PLR": 0.443, "p_LOR": 0.614},
])
 
# Apply Bonferroni correction 
for predictor in ["PLR", "LOR"]:
    col = f"p_{predictor}"
    _, p_corrected, _, _ = multipletests(results[col], alpha=0.05, method="bonferroni")
    results[f"p_{predictor}_corrected"] = p_corrected
    results[f"{predictor}_significant"] = p_corrected < 0.05
 
n_tests = len(results)
alpha_corrected = 0.05 / n_tests
 
# Print in a nice way
print("=" * 65)
print("Bonferroni correction — actual days")
print(f"Outcomes: FOUR, GCS, SECONDS  |  Horizons: t0, t1, t3, t7")
print(f"Total tests per predictor: {n_tests}  |  Corrected α: {alpha_corrected:.4f}")
print("=" * 65)
 
for predictor in ["LOR", "PLR"]:
    print(f"\nz_{predictor}")
    print("-" * 55)
    print(f"  {'Outcome':<10} {'Horizon':<10} {'Raw p':<12} {'Corrected p':<14} {'Sig?'}")
    print(f"  {'-'*8:<10} {'-'*7:<10} {'-'*9:<12} {'-'*11:<14} {'-'*4}")
    for _, row in results.iterrows():
        raw       = row[f"p_{predictor}"]
        corrected = row[f"p_{predictor}_corrected"]
        sig       = row[f"{predictor}_significant"]
        raw_str   = "< 0.001" if raw < 0.001 else f"{raw:.3f}"
        cor_str   = "< 0.001" if corrected < 0.001 else f"{corrected:.3f}"
        sig_str   = "YES *" if sig else "no"
        print(f"  {row['outcome']:<10} {row['horizon']:<10} {raw_str:<12} {cor_str:<14} {sig_str}")
 
print("\n" + "=" * 65)
print("Summary")
print("-" * 65)
for predictor in ["LOR", "PLR"]:
    n_sig = results[f"{predictor}_significant"].sum()
    print(f"  z_{predictor}: {n_sig}/{n_tests} significant after correction")
print("=" * 65)

Bonferroni correction — actual days
Outcomes: FOUR, GCS, SECONDS  |  Horizons: t0, t1, t3, t7
Total tests per predictor: 12  |  Corrected α: 0.0042

z_LOR
-------------------------------------------------------
  Outcome    Horizon    Raw p        Corrected p    Sig?
  --------   -------    ---------    -----------    ----
  FOUR       t0         0.855        1.000          no
  FOUR       t1         0.052        0.624          no
  FOUR       t3         0.064        0.768          no
  FOUR       t7         0.679        1.000          no
  GCS        t0         0.958        1.000          no
  GCS        t1         0.012        0.144          no
  GCS        t3         0.108        1.000          no
  GCS        t7         0.382        1.000          no
  SECONDS    t0         0.618        1.000          no
  SECONDS    t1         0.008        0.096          no
  SECONDS    t3         0.773        1.000          no
  SECONDS    t7         0.614        1.000          no

z_PLR
--------